In [1]:
import os
import numpy as np
from matplotlib import pyplot as plt

import torch

In [2]:
# Model Init
# Prompt Init
# Data Loader & Chunking Init
# RAG Pipeline Init
# Inference

### Model

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import pipeline
from langchain import HuggingFacePipeline

In [4]:
model_name = "HuggingFaceTB/SmolLM2-360M-Instruct" # Qwen/Qwen2.5-3B-Instruct, HuggingFaceTB/SmolLM2-1.7B-Instruct

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

pipe = pipeline(task = "text-generation", model=model, tokenizer=tokenizer, max_new_tokens=128, temperature=0.01, do_sample=True)
hug_pipe = HuggingFacePipeline(pipeline=pipe)

model.safetensors:   9%|8         | 62.9M/724M [00:00<?, ?B/s]

c:\Users\citak\anaconda3\envs\pytorch\lib\site-packages\huggingface_hub\file_download.py:147: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\citak\.cache\huggingface\hub\models--HuggingFaceTB--SmolLM2-360M-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

C:\Users\citak\AppData\Local\Temp\ipykernel_17032\573512347.py:11: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFacePipeline`.
  hug_pipe = HuggingFacePipeline(pipeline=pipe)


### Prompt

In [5]:
from langchain.prompts import PromptTemplate

In [6]:
prompt = """
You are a helpful assistant. Answer the user's question clearly, without using fluffy words.

Question:
{question}

Answer:
"""

prompt_rag = """
You are a helpful assistant. Answer the user's question based on the provided context below.
If you can't infer answer from the context, just return 'I don't know'. 
Your answer must be clear, not include fluffy words and inside of json block.

Context:
{context}

Question:
{question}

Answer:
'''json
"""

In [7]:
prompt_template = PromptTemplate(
    input_variables=["question"],
    template=prompt,
)

prompt_template_rag = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_rag,
)

In [8]:
prompt_formatted = prompt_template.format(
    question= "How are you?"
)

prompt_rag_formatted = prompt_template_rag.format(
    context = "Real Madrid and Barcelone are two big clubs of Spanish Footbal League.",
    question = "Which country does Barcelone play for?"
)

In [9]:
response = hug_pipe.invoke(prompt_formatted)
print(response)

c:\Users\citak\anaconda3\envs\pytorch\lib\site-packages\transformers\models\llama\modeling_llama.py:602: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(



You are a helpful assistant. Answer the user's question clearly, without using fluffy words.

Question:
How are you?

Answer:
I am doing well, thank you for asking.


In [10]:
response = hug_pipe.invoke(prompt_rag_formatted)
print(response)


You are a helpful assistant. Answer the user's question based on the provided context below.
If you can't infer answer from the context, just return 'I don't know'. 
Your answer must be clear, not include fluffy words and inside of json block.

Context:
Real Madrid and Barcelone are two big clubs of Spanish Footbal League.

Question:
Which country does Barcelone play for?

Answer:
'''json
{
  "clubs": [
    {
      "name": "Real Madrid",
      "country": "Spain"
    },
    {
      "name": "Barcelone",
      "country": "Spain"
    }
  ]
}
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"
"



### Data Loader & Chunking Init

In [11]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [12]:
data_folder = "data/"
file_names = os.listdir(data_folder)
pages = []

for pdf_name in file_names:
    pdf_path = os.path.join(data_folder, pdf_name)

    loader = PyPDFLoader(pdf_path)
    
    for page in loader.load():
        pages.append(page)

Ignoring wrong pointing object 13 0 (offset 0)


In [13]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(pages)

In [14]:
print(len(pages), len(all_splits))

28 100


### RAG Pipeline Init

In [15]:
# Vector db
# RAG Pipeline

In [16]:
from langchain.embeddings import HuggingFaceEmbeddings

import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

from uuid import uuid4

In [17]:
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

C:\Users\citak\AppData\Local\Temp\ipykernel_17032\3234083693.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(


In [18]:
sample_embedding = embeddings.embed_query("Erol was here")
len(sample_embedding)

index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

In [19]:
vector_db = FAISS.from_documents(all_splits, embeddings)
vector_db_retriever = vector_db.as_retriever(search_kwargs={"k": 3})

### Chain

In [20]:
from langchain.chains import RetrievalQA

In [21]:
qa_chain = RetrievalQA.from_chain_type(
    llm=hug_pipe,
    retriever=vector_db_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template_rag},
    return_source_documents=True
)

### Inference

In [22]:
question = "What are the ethical rules of Apple?"
result = qa_chain.invoke({"query": question})


print("\nAnswer:", result['result'])
print("\n--- Retrieved Docs ---")
for i, doc in enumerate(result["source_documents"]):
    print(f"[{i+1}] {doc.page_content[:200]}...\n")


Answer: 
You are a helpful assistant. Answer the user's question based on the provided context below.
If you can't infer answer from the context, just return 'I don't know'. 
Your answer must be clear, not include fluffy words and inside of json block.

Context:
to you. 
Any waiver of this Policy for our directors, executive officers, or principal accounting officer may be made only by our Board 
of Directors, and will be disclosed as required by law or applicable listing rules.
The way we do business worldwide
At Apple, we are committed to demonstrating that business can and should be a force for good. 
Achieving that takes innovation, collaboration, and a focus on serving others. 
It also means leading with our values—accessibility, education, environment, inclusion and diversity, 
privacy, racial equity and justice, and supplier responsibility. Our Business Conduct Policy is 
foundational to how we do business and how we put our values into practice each and every day.
Apple conduc